# Codelists

- Preparing the codelists for phenotype extraction including:

  - Pre-processing
  - READ2 to CTV3 mapping


In [1]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 500)

In [2]:
import os
import sys

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [3]:
from env.parameters import P

In [ ]:
P.codelist_path

In [5]:
# Codelist where read2 is different form ctv3
df_read_diff = pd.read_csv(f'''{P.codelist_path}/diff_read2_ctv3_for_codelists.csv''')
df_read_diff = df_read_diff.rename(columns= {"READV2_CODE":"code_clean", "READV3_CODE":"READ3"})

In [6]:
df_read_diff.head()

,code_clean,READV2_DESC,READ3,TERMV3_DESC
0,01...,Top managers,XE0Nq,Top manager
1,01...,Top managers,XaBKm,Administrator - top
2,013..,Top managers - business,XE0Nr,Top manager - business
3,013..,Top managers - business,XaBNP,Business - top men
4,013..,Top managers - business,XaBNO,Director - large company


In [7]:
path = f'''{P.codelist_path}/curated'''

# Motor Neuron


In [9]:
df = pd.read_csv(f'''{P.codelist_path}/Motor_Neuron_phenotype_PH62_ver_124_concepts_20231228T125837.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,F15y.00,Other anterior horn cell disease,Read codes v2,C852,2843,Motor neuron disease - Primary care,PH62,124,Motor neuron disease,"[{""Disease"": ""Motor neuron disease"", ""Medcode""..."
1,F15z.00,Anterior horn cell disease NOS,Read codes v2,C852,2843,Motor neuron disease - Primary care,PH62,124,Motor neuron disease,"[{""Disease"": ""Motor neuron disease"", ""Medcode""..."
2,F152z00,Motor neurone disease NOS,Read codes v2,C852,2843,Motor neuron disease - Primary care,PH62,124,Motor neuron disease,"[{""Disease"": ""Motor neuron disease"", ""Medcode""..."
3,F15..00,Anterior horn cell disease,Read codes v2,C852,2843,Motor neuron disease - Primary care,PH62,124,Motor neuron disease,"[{""Disease"": ""Motor neuron disease"", ""Medcode""..."
4,F152000,Amyotrophic lateral sclerosis,Read codes v2,C852,2843,Motor neuron disease - Primary care,PH62,124,Motor neuron disease,"[{""Disease"": ""Motor neuron disease"", ""Medcode""..."


In [10]:
df.groupby(by="coding_system").agg("count")["code"]

coding_system
ICD10 codes       1
Med codes        12
Read codes v2    12
Name: code, dtype: int64

In [11]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})

In [12]:
df.groupby(by="vocab").agg("count")["code"]


vocab
ICD10     1
Read2    12
Name: code, dtype: int64

In [13]:
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
    #.str.lower()

In [14]:
df["pheno_name"] = "motor_neuron"
df["info"] = "PH62"

In [15]:
df_sel =df[["pheno_name","code_clean", "vocab","description",  "info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,motor_neuron,F15y.00,Read2,Other anterior horn cell disease,PH62
1,motor_neuron,F15z.00,Read2,Anterior horn cell disease NOS,PH62
2,motor_neuron,F152z00,Read2,Motor neurone disease NOS,PH62
3,motor_neuron,F15..00,Read2,Anterior horn cell disease,PH62
4,motor_neuron,F152000,Read2,Amyotrophic lateral sclerosis,PH62


In [16]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)

vocab
Read2    12
ICD10     1
Name: count, dtype: int64

In [17]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [20]:
df_sel = df_sel[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]

In [21]:
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,motor_neuron,F15y.00,Read2,Other anterior horn cell disease,PH62
1,motor_neuron,F15z.00,Read2,Anterior horn cell disease NOS,PH62
2,motor_neuron,F152z00,Read2,Motor neurone disease NOS,PH62
3,motor_neuron,F15..00,Read2,Anterior horn cell disease,PH62
4,motor_neuron,F152000,Read2,Amyotrophic lateral sclerosis,PH62


In [22]:
df_sel.to_csv(f'''{path}/motor_neuron.csv''', index=False)

# Parkinson

In [23]:
# codelist
df = pd.read_csv(f'''{P.codelist_path}/Parkinson_phenotype_PH77_ver_154_concepts_20231228T125909.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,Eu02300,[X]Dementia in Parkinson's disease,Read codes v2,C896,2931,Parkinson's disease - Primary care,PH77,154,Parkinson's disease,"[{""Disease"": ""Parkinson's disease"", ""Medcode"":..."
1,147F.00,History of Parkinson's disease,Read codes v2,C896,2931,Parkinson's disease - Primary care,PH77,154,Parkinson's disease,"[{""Disease"": ""Parkinson's disease"", ""Medcode"":..."
2,F12z.00,Parkinson's disease NOS,Read codes v2,C896,2931,Parkinson's disease - Primary care,PH77,154,Parkinson's disease,"[{""Disease"": ""Parkinson's disease"", ""Medcode"":..."
3,F11x900,Cerebral degeneration in Parkinson's disease,Read codes v2,C896,2931,Parkinson's disease - Primary care,PH77,154,Parkinson's disease,"[{""Disease"": ""Parkinson's disease"", ""Medcode"":..."
4,F12..00,Parkinson's disease,Read codes v2,C896,2931,Parkinson's disease - Primary care,PH77,154,Parkinson's disease,"[{""Disease"": ""Parkinson's disease"", ""Medcode"":..."


In [24]:
df.groupby(by="coding_system").agg("count")["code"]

coding_system
ICD10 codes      2
Med codes        6
Read codes v2    6
Name: code, dtype: int64

In [25]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})

In [26]:
df.groupby(by="vocab").agg("count")["code"]


vocab
ICD10    2
Read2    6
Name: code, dtype: int64

In [27]:
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
    #.str.lower()

In [28]:
df["pheno_name"] = "parkinson"
df["info"] = "PH77"

In [29]:
df_sel =df[["pheno_name","code_clean", "vocab","description",  "info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,parkinson,Eu02300,Read2,[X]Dementia in Parkinson's disease,PH77
1,parkinson,147F.00,Read2,History of Parkinson's disease,PH77
2,parkinson,F12z.00,Read2,Parkinson's disease NOS,PH77
3,parkinson,F11x900,Read2,Cerebral degeneration in Parkinson's disease,PH77
4,parkinson,F12..00,Read2,Parkinson's disease,PH77


In [30]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)

vocab
Read2    6
ICD10    2
Name: count, dtype: int64

In [31]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [32]:
df_sel = df_sel[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]

In [33]:
df_sel.to_csv(f'''{path}/parkinson.csv''', index=False)

# AAA

In [35]:
# codelist
df = pd.read_csv(f'''{P.codelist_path}/abdominal_aortic_aneurysm_phenotype_PH34_ver_68_concepts_20240309T212846.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,7A11000,Emerg repl aneurysm bifurc aorta by anast aort...,Read codes v2,C783,2705,Abdominal aortic aneurysm - Primary care,PH34,68,Abdominal aortic aneurysm,"[{""Disease"": ""Abdominal aortic aneurysm"", ""Med..."
1,14AE.00,H/O: aortic aneurysm,Read codes v2,C783,2705,Abdominal aortic aneurysm - Primary care,PH34,68,Abdominal aortic aneurysm,"[{""Disease"": ""Abdominal aortic aneurysm"", ""Med..."
2,G714.11,AAA - Abdominal aortic aneurysm without mentio...,Read codes v2,C783,2705,Abdominal aortic aneurysm - Primary care,PH34,68,Abdominal aortic aneurysm,"[{""Disease"": ""Abdominal aortic aneurysm"", ""Med..."
3,7A13z00,Emergency replacement of aneurysmal segment of...,Read codes v2,C783,2705,Abdominal aortic aneurysm - Primary care,PH34,68,Abdominal aortic aneurysm,"[{""Disease"": ""Abdominal aortic aneurysm"", ""Med..."
4,7A13400,Emerg replace aneurysm abdom aorta by anast ao...,Read codes v2,C783,2705,Abdominal aortic aneurysm - Primary care,PH34,68,Abdominal aortic aneurysm,"[{""Disease"": ""Abdominal aortic aneurysm"", ""Med..."


In [36]:
df.groupby(by="coding_system").agg("count")["code"]

coding_system
ICD10 codes       3
Med codes        15
Read codes v2    15
Name: code, dtype: int64

In [37]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})

In [38]:
df.groupby(by="vocab").agg("count")["code"]


vocab
ICD10     3
Read2    15
Name: code, dtype: int64

In [39]:
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
    #.str.lower()

In [40]:
df["pheno_name"] = "aaa"
df["info"] = "PH34"

In [41]:
df_sel =df[["pheno_name","code_clean", "vocab","description",  "info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,aaa,7A11000,Read2,Emerg repl aneurysm bifurc aorta by anast aort...,PH34
1,aaa,14AE.00,Read2,H/O: aortic aneurysm,PH34
2,aaa,G714.11,Read2,AAA - Abdominal aortic aneurysm without mentio...,PH34
3,aaa,7A13z00,Read2,Emergency replacement of aneurysmal segment of...,PH34
4,aaa,7A13400,Read2,Emerg replace aneurysm abdom aorta by anast ao...,PH34


In [42]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)

vocab
Read2    15
ICD10     3
Name: count, dtype: int64

In [43]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [44]:
df_sel.to_csv(f'''{path}/aaa.csv''', index=False)

# Anxiety

In [10]:
df = pd.read_csv(f'''{P.codelist_path}/Anxiety_phenotype_PH1113_ver_2453_concepts_20240217T235545.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,E2000,anxiety state unspecified,Read codes v2,C2921,10162,Anxiety Diagnosis (Core)- Read Codes,PH1113,2453,Anxiety- Phenotype,NaN
1,Eu41z,[x]anxiety disord unspecified | [x]anxiety dis...,Read codes v2,C2921,10162,Anxiety Diagnosis (Core)- Read Codes,PH1113,2453,Anxiety- Phenotype,NaN
2,Eu932,[x]social anx dis childhood | [x]social anxiet...,Read codes v2,C2921,10162,Anxiety Diagnosis (Core)- Read Codes,PH1113,2453,Anxiety- Phenotype,NaN
3,E2021,agoraphobia with panic attacks,Read codes v2,C2921,10162,Anxiety Diagnosis (Core)- Read Codes,PH1113,2453,Anxiety- Phenotype,NaN
4,E2D00,child/adolesc.overanxious.dis. | childhood and...,Read codes v2,C2921,10162,Anxiety Diagnosis (Core)- Read Codes,PH1113,2453,Anxiety- Phenotype,NaN


In [11]:
df.groupby(by="coding_system").agg("count")["code"]

coding_system
ICD10 codes       22
Read codes v2    335
Name: code, dtype: int64

In [12]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})

In [13]:
df.groupby(by="vocab").agg("count")["code"]


vocab
ICD10     22
Read2    335
Name: code, dtype: int64

In [14]:
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
    #.str.lower()

In [15]:
df["pheno_name"] = "Anxiety"
df["info"] = "PH1113_v2453"

In [16]:
df_sel =df[["pheno_name","code_clean", "vocab","description",  "info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Anxiety,E2000,Read2,anxiety state unspecified,PH1113_v2453
1,Anxiety,Eu41z,Read2,[x]anxiety disord unspecified | [x]anxiety dis...,PH1113_v2453
2,Anxiety,Eu932,Read2,[x]social anx dis childhood | [x]social anxiet...,PH1113_v2453
3,Anxiety,E2021,Read2,agoraphobia with panic attacks,PH1113_v2453
4,Anxiety,E2D00,Read2,child/adolesc.overanxious.dis. | childhood and...,PH1113_v2453


In [17]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)

vocab
Read2    335
ICD10     22
Name: count, dtype: int64

In [18]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,Anxiety,Read3,[x]anxiety disord unspecified | [x]anxiety dis...,PH1113_v2453,"[X]Anxiety disorder, unspecified",E200.,Anxiety disorder
1,Anxiety,Read3,[x]social anx dis childhood | [x]social anxiet...,PH1113_v2453,[X]Social anxiety disorder of childhood,XE1aD,Childhood social anxiety disorder
2,Anxiety,Read3,[x]social anx dis childhood | [x]social anxiet...,PH1113_v2453,[X]Social anxiety disorder of childhood,XE1aD,Childhood avoidant disorder
3,Anxiety,Read3,[x]social anx dis childhood | [x]social anxiet...,PH1113_v2453,[X]Social anxiety disorder of childhood,XE1aD,Childhood social anxiety disorder
4,Anxiety,Read3,[x]social anx dis childhood | [x]social anxiet...,PH1113_v2453,[X]Social anxiety disorder of childhood,XE1aD,Childhood avoidant disorder


In [19]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]

In [20]:
df_read_new.shape

(78, 5)

In [21]:
df_sel_final = pd.concat([df_sel, df_read_new])

In [22]:
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,Anxiety,E2000,Read2,anxiety state unspecified,PH1113_v2453
1,Anxiety,Eu41z,Read2,[x]anxiety disord unspecified | [x]anxiety dis...,PH1113_v2453
2,Anxiety,Eu932,Read2,[x]social anx dis childhood | [x]social anxiet...,PH1113_v2453
3,Anxiety,E2021,Read2,agoraphobia with panic attacks,PH1113_v2453
4,Anxiety,E2D00,Read2,child/adolesc.overanxious.dis. | childhood and...,PH1113_v2453


In [23]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    335
Read3     78
ICD10     22
Name: count, dtype: int64

In [24]:
df_sel_final.to_csv(f'''{path}/anxiety_updated.csv''', index=False)

# AMI

In [349]:

df = pd.read_csv(f'''{P.codelist_path}/AMI_phenotype_PH949_ver_2127_concepts_20231109T105016.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,I23,Certain current complications following acute ...,ICD10 codes,C2739,7260,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,I21,Acute myocardial infarction,ICD10 codes,C2739,7260,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,I22,Subsequent myocardial infarction,ICD10 codes,C2739,7260,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
3,70422006,Acute subendocardial infarction,SNOMED CT codes,C2740,7261,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
4,703209002,Subsequent ST segment elevation myocardial inf...,SNOMED CT codes,C2740,7261,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"


In [350]:
# Test

In [351]:
df.groupby(by="coding_system").agg("count")["code"]

coding_system
ICD10 codes          5
Read codes v2       60
SNOMED  CT codes    75
Name: code, dtype: int64

In [352]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})

In [353]:
df.groupby(by="vocab").agg("count")["code"]


vocab
ICD10      5
Read2     60
SNOMED    75
Name: code, dtype: int64

In [354]:
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
    #.str.lower()

In [355]:
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes,vocab,code_clean
0,I23,Certain current complications following acute ...,ICD10 codes,C2739,7260,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]",ICD10,I23
1,I21,Acute myocardial infarction,ICD10 codes,C2739,7260,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]",ICD10,I21
2,I22,Subsequent myocardial infarction,ICD10 codes,C2739,7260,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]",ICD10,I22
3,70422006,Acute subendocardial infarction,SNOMED CT codes,C2740,7261,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]",SNOMED,70422006
4,703209002,Subsequent ST segment elevation myocardial inf...,SNOMED CT codes,C2740,7261,CCU002_01 Acute myocardial infarction - Phenot...,PH949,2127,CCU002_01 Acute myocardial infarction,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]",SNOMED,703209002


In [356]:
df["pheno_name"] = "AMI"
df["info"] = "PH949_v2127"

In [357]:
df['vocab'].value_counts(dropna=False)

vocab
SNOMED    75
Read2     60
ICD10      5
Name: count, dtype: int64

In [358]:
df_sel =df[["pheno_name","code_clean", "vocab","description",  "info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,AMI,I23,ICD10,Certain current complications following acute ...,PH949_v2127
1,AMI,I21,ICD10,Acute myocardial infarction,PH949_v2127
2,AMI,I22,ICD10,Subsequent myocardial infarction,PH949_v2127
3,AMI,70422006,SNOMED,Acute subendocardial infarction,PH949_v2127
4,AMI,703209002,SNOMED,Subsequent ST segment elevation myocardial inf...,PH949_v2127


In [359]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    60
ICD10     5
Name: count, dtype: int64

In [360]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,AMI,Read3,Acute transmural myocardial infarction of unsp...,PH949_v2127,Acute transmural myocardial infarction of unsp...,Gyu34,[X]Acute transmural myocardial infarction of u...
1,AMI,Read3,Acute non-Q wave infarction ...,PH949_v2127,Acute non-Q wave infarction,XaAzi,Acute non-Q wave infarction
2,AMI,Read3,Diab mellit insulin-glucose infus acute myocar...,PH949_v2127,Diabetes mellitus insulin-glucose infusion in ...,XaFx7,Diabetes mellitus insulin-glucose infusion in ...
3,AMI,Read3,Postoperative subendocardial myocardial infarc...,PH949_v2127,Postoperative subendocardial myocardial infarc...,XaD2h,Postoperative subendocardial myocardial infarc...
4,AMI,Read3,Old myocardial infarction ...,PH949_v2127,Old myocardial infarction,XE2aA,Old myocardial infarction


In [361]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]

In [362]:
df_read_new.shape

(33, 5)

In [363]:
df_sel_final = pd.concat([df_sel, df_read_new])

In [364]:
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,AMI,I23,ICD10,Certain current complications following acute ...,PH949_v2127
1,AMI,I21,ICD10,Acute myocardial infarction,PH949_v2127
2,AMI,I22,ICD10,Subsequent myocardial infarction,PH949_v2127
78,AMI,I241,ICD10,Dressler's syndrome,PH949_v2127
79,AMI,I252,ICD10,Old myocardial infarction,PH949_v2127


In [365]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    60
Read3    33
ICD10     5
Name: count, dtype: int64

In [366]:
df_sel_final.to_csv(f'''{path}/ami_updated.csv''', index=False)

In [367]:
test = pd.read_csv(f'''{path}/ami_updated.csv''')
test.head()

,pheno_name,code_clean,vocab,description,info
0,AMI,I23,ICD10,Certain current complications following acute ...,PH949_v2127
1,AMI,I21,ICD10,Acute myocardial infarction,PH949_v2127
2,AMI,I22,ICD10,Subsequent myocardial infarction,PH949_v2127
3,AMI,I241,ICD10,Dressler's syndrome,PH949_v2127
4,AMI,I252,ICD10,Old myocardial infarction,PH949_v2127


# Asthma

In [368]:

df = pd.read_csv(f'''{P.codelist_path}/asthma_phenotype_PH12_ver_24_concepts_20231225T224912.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,663d.00,Emergency asthma admission since last appointment,Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""3033971010"", ""S..."
1,1788.00,Asthma trigger - cold air,Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""305961000000113..."
2,663N000,Asthma causing night waking,Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""264541019"", ""SN..."
3,679J100,Health education - structured asthma discussion,Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""117654100000011..."
4,8H2P.00,"Emergency admission, asthma",Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""3082895019"", ""S..."


In [369]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes           4
ICD11 codes           3
Read codes v2       165
SNOMED  CT codes    150
UKBioBank codes       1
Name: code, dtype: int64

In [370]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})


In [371]:
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes,vocab,code_clean
0,663d.00,Emergency asthma admission since last appointment,Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""3033971010"", ""S...",Read2,663d.00
1,1788.00,Asthma trigger - cold air,Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""305961000000113...",Read2,1788.00
2,663N000,Asthma causing night waking,Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""264541019"", ""SN...",Read2,663N000
3,679J100,Health education - structured asthma discussion,Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""117654100000011...",Read2,679J100
4,8H2P.00,"Emergency admission, asthma",Read codes v2,C733,2605,Asthma - Primary care,PH12,24,Asthma,"[{""SNOMED CT Description ID"": ""3082895019"", ""S...",Read2,8H2P.00


In [372]:
df["pheno_name"] = "asthma"
df["info"] = "PH12_v24"

In [373]:
df['vocab'].value_counts(dropna=False)


vocab
Read2     165
SNOMED    150
NaN         4
ICD10       4
Name: count, dtype: int64

In [374]:
df_sel =df[["pheno_name","code_clean", "vocab", "description", "info"]]

df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)

vocab
Read2    165
ICD10      4
Name: count, dtype: int64

In [375]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [376]:
df_sel.to_csv(f'''{path}/asthma.csv''', index=False)


# Cancer

In [377]:

df = pd.read_csv(f'''{P.codelist_path}/Cancer_phenotype_PH960_ver_2138_concepts_20231115T192850.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,C10,C10 Malignant neoplasm of oropharynx,ICD10 codes,C2765,7286,CCU002_01 Cancer - Phenotypes - England hospit...,PH960,2138,CCU002_01 Cancer,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,C38,"C38 Malignant neoplasm of heart, mediastinum a...",ICD10 codes,C2765,7286,CCU002_01 Cancer - Phenotypes - England hospit...,PH960,2138,CCU002_01 Cancer,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,C62,C62 Malignant neoplasm of testis,ICD10 codes,C2765,7286,CCU002_01 Cancer - Phenotypes - England hospit...,PH960,2138,CCU002_01 Cancer,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
3,C32,C32 Malignant neoplasm of larynx,ICD10 codes,C2765,7286,CCU002_01 Cancer - Phenotypes - England hospit...,PH960,2138,CCU002_01 Cancer,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
4,C56,C56 Malignant neoplasm of ovary,ICD10 codes,C2765,7286,CCU002_01 Cancer - Phenotypes - England hospit...,PH960,2138,CCU002_01 Cancer,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"


In [378]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes           70
Read codes v2       1667
SNOMED  CT codes    1918
Name: code, dtype: int64

In [379]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})

In [380]:
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
    #.str.lower()

In [381]:
df["pheno_name"] = "Cancer"
df["info"] = "PH960_v2138"

In [382]:
df['vocab'].value_counts(dropna=False)


vocab
SNOMED    1918
Read2     1667
ICD10       70
Name: count, dtype: int64

In [383]:
df_sel =df[["pheno_name","code_clean", "vocab", "description", "info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Cancer,C10,ICD10,C10 Malignant neoplasm of oropharynx,PH960_v2138
1,Cancer,C38,ICD10,"C38 Malignant neoplasm of heart, mediastinum a...",PH960_v2138
2,Cancer,C62,ICD10,C62 Malignant neoplasm of testis,PH960_v2138
3,Cancer,C32,ICD10,C32 Malignant neoplasm of larynx,PH960_v2138
4,Cancer,C56,ICD10,C56 Malignant neoplasm of ovary,PH960_v2138


In [384]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    1667
ICD10      70
Name: count, dtype: int64

In [385]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,Cancer,Read3,Fast track referral for suspected childrens tu...,PH960_v2138,Fast track referral for suspected childrens tu...,XaMeD,Fast track referral for suspected childrens tu...
1,Cancer,Read3,Malignant neoplasm of palatopharyngeal arch ...,PH960_v2138,Malignant neoplasm of palatopharyngeal arch,B062.,Malignant tumour of palatopharyngeal arch
2,Cancer,Read3,Malignant neoplasm of palatopharyngeal arch ...,PH960_v2138,Malignant neoplasm of palatopharyngeal arch,B062.,Malignant tumour of tonsillar pillar
3,Cancer,Read3,Malignant neoplasm of canthus ...,PH960_v2138,Malignant neoplasm of canthus,XaCKx,Malignant neoplasm of canthus
4,Cancer,Read3,"Malignant neoplasm of lip unspecified, buccal ...",PH960_v2138,"Malignant neoplasm of lip unspecified, buccal ...",B0042,"Malignant neoplasm of lip unspecified, mucosa"


In [386]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]

In [387]:
df_read_new.shape

(410, 5)

In [388]:
df_sel_final = pd.concat([df_sel, df_read_new])

In [389]:
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,Cancer,C10,ICD10,C10 Malignant neoplasm of oropharynx,PH960_v2138
1,Cancer,C38,ICD10,"C38 Malignant neoplasm of heart, mediastinum a...",PH960_v2138
2,Cancer,C62,ICD10,C62 Malignant neoplasm of testis,PH960_v2138
3,Cancer,C32,ICD10,C32 Malignant neoplasm of larynx,PH960_v2138
4,Cancer,C56,ICD10,C56 Malignant neoplasm of ovary,PH960_v2138


In [390]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    1667
Read3     410
ICD10      70
Name: count, dtype: int64

In [391]:
df_sel_final.to_csv(f'''{path}/cancer_updated.csv''', index=False)


# Cardiomyopathy

In [392]:
df = pd.read_csv(f'''{P.codelist_path}/Cardiomyopathy_phenotype_PH961_ver_2139_concepts_20231109T110043.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,O90.3,Cardiomyopathy in the puerperium,ICD10 codes,C2769,7290,CCU002_01 Cardiomyopathy - Phenotypes - Englan...,PH961,2139,CCU002_01 Cardiomyopathy,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,I43,Cardiomyopathy in diseases classified elsewhere,ICD10 codes,C2769,7290,CCU002_01 Cardiomyopathy - Phenotypes - Englan...,PH961,2139,CCU002_01 Cardiomyopathy,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,I42.5,Other restrictive cardiomyopathy,ICD10 codes,C2769,7290,CCU002_01 Cardiomyopathy - Phenotypes - Englan...,PH961,2139,CCU002_01 Cardiomyopathy,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
3,I25.5,Ischaemic cardiomyopathy,ICD10 codes,C2769,7290,CCU002_01 Cardiomyopathy - Phenotypes - Englan...,PH961,2139,CCU002_01 Cardiomyopathy,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
4,I42.0,Dilated cardiomyopathy,ICD10 codes,C2769,7290,CCU002_01 Cardiomyopathy - Phenotypes - Englan...,PH961,2139,CCU002_01 Cardiomyopathy,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"


In [393]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes          9
Read codes v2       31
SNOMED  CT codes     3
Name: code, dtype: int64

In [394]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
df["pheno_name"] = "Cardiomyopathy"
df["info"] = "PH961_v2139"
df['vocab'].value_counts(dropna=False)


vocab
Read2     31
ICD10      9
SNOMED     3
Name: count, dtype: int64

In [395]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Cardiomyopathy,O903,ICD10,Cardiomyopathy in the puerperium,PH961_v2139
1,Cardiomyopathy,I43,ICD10,Cardiomyopathy in diseases classified elsewhere,PH961_v2139
2,Cardiomyopathy,I425,ICD10,Other restrictive cardiomyopathy,PH961_v2139
3,Cardiomyopathy,I255,ICD10,Ischaemic cardiomyopathy,PH961_v2139
4,Cardiomyopathy,I420,ICD10,Dilated cardiomyopathy,PH961_v2139


In [396]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    31
ICD10     9
Name: count, dtype: int64

In [397]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,Cardiomyopathy,Read3,Tachycardiomyopathy ...,PH961_v2139,Tachycardiomyopathy,XaXNg,Tachycardiomyopathy
1,Cardiomyopathy,Read3,Tachycardiomyopathy ...,PH961_v2139,Tachycardiomyopathy,XaXNg,Tachycardia-induced cardiomyopathy
2,Cardiomyopathy,Read3,Secondary cardiomyopathy NOS ...,PH961_v2139,Secondary cardiomyopathy NOS,XE0V1,Secondary cardiomyopathy NOS
3,Cardiomyopathy,Read3,Secondary cardiomyopathy NOS ...,PH961_v2139,Secondary cardiomyopathy NOS,X201Z,Secondary dilated cardiomyopathy
4,Cardiomyopathy,Read3,Amyloid cardiomyopathy ...,PH961_v2139,Amyloid cardiomyopathy,XaZcx,Amyloid cardiomyopathy


In [398]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]

In [399]:
df_read_new.shape

(16, 5)

In [400]:
df_sel_final = pd.concat([df_sel, df_read_new])

In [401]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    31
Read3    16
ICD10     9
Name: count, dtype: int64

In [402]:
df_sel_final.to_csv(f'''{path}/cardiomyopathy_updated.csv''', index=False)


# COPD

In [403]:
df = pd.read_csv(f'''{P.codelist_path}/COPD_phenotype_PH951_ver_2129_concepts_20231115T193027.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,J42,Unspecified chronic bronchitis,ICD10 codes,C2746,7267,CCU002_01 Chronic obstructive pulmonary diseas...,PH951,2129,CCU002_01 Chronic obstructive pulmonary diseas...,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,J41,Simple and mucopurulent chronic bronchitis,ICD10 codes,C2746,7267,CCU002_01 Chronic obstructive pulmonary diseas...,PH951,2129,CCU002_01 Chronic obstructive pulmonary diseas...,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,J40,Bronchitis not specified as acute or chronic,ICD10 codes,C2746,7267,CCU002_01 Chronic obstructive pulmonary diseas...,PH951,2129,CCU002_01 Chronic obstructive pulmonary diseas...,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
3,J44,Other chronic obstructive pulmonary disease,ICD10 codes,C2746,7267,CCU002_01 Chronic obstructive pulmonary diseas...,PH951,2129,CCU002_01 Chronic obstructive pulmonary diseas...,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
4,J43,Emphysema,ICD10 codes,C2746,7267,CCU002_01 Chronic obstructive pulmonary diseas...,PH951,2129,CCU002_01 Chronic obstructive pulmonary diseas...,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"


In [404]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes          5
Read codes v2       50
SNOMED  CT codes    41
Name: code, dtype: int64

In [405]:
df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes":"SNOMED", "Read codes v2":"Read2"})
df["code_clean"]  =df["code"]
df.loc[df["vocab"]=="ICD10", "code_clean"] = df["code_clean"].str.replace(".","")
df["pheno_name"] = "COPD"
df["info"] = "PH951_v2129"
df['vocab'].value_counts(dropna=False)

vocab
Read2     50
SNOMED    41
ICD10      5
Name: count, dtype: int64

In [406]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,COPD,J42,ICD10,Unspecified chronic bronchitis,PH951_v2129
1,COPD,J41,ICD10,Simple and mucopurulent chronic bronchitis,PH951_v2129
2,COPD,J40,ICD10,Bronchitis not specified as acute or chronic,PH951_v2129
3,COPD,J44,ICD10,Other chronic obstructive pulmonary disease,PH951_v2129
4,COPD,J43,ICD10,Emphysema,PH951_v2129


In [407]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    50
ICD10     5
Name: count, dtype: int64

In [408]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,COPD,Read3,Eosinophilic bronchitis ...,PH951_v2129,Eosinophilic bronchitis,Xaa7C,Eosinophilic bronchitis
1,COPD,Read3,History of chronic obstructive pulmonary disea...,PH951_v2129,H/O: chr.obstr. airway disease,XE2Pp,NaN
2,COPD,Read3,History of chronic obstructive pulmonary disea...,PH951_v2129,H/O: chr.obstr. airway disease,XE2Pp,History of chronic obstructive pulmonary disease
3,COPD,Read3,History of chronic obstructive pulmonary disea...,PH951_v2129,H/O: chr.obstr. airway disease,XM1R4,H/O: bronchitis
4,COPD,Read3,History of chronic obstructive pulmonary disea...,PH951_v2129,H/O: chr.obstr. airway disease,XE2Pp,History of chronic obstructive pulmonary disease


In [409]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]
df_read_new.shape

(24, 5)

In [410]:
df_sel_final = pd.concat([df_sel, df_read_new])
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,COPD,J42,ICD10,Unspecified chronic bronchitis,PH951_v2129
1,COPD,J41,ICD10,Simple and mucopurulent chronic bronchitis,PH951_v2129
2,COPD,J40,ICD10,Bronchitis not specified as acute or chronic,PH951_v2129
3,COPD,J44,ICD10,Other chronic obstructive pulmonary disease,PH951_v2129
4,COPD,J43,ICD10,Emphysema,PH951_v2129


In [411]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    50
Read3    24
ICD10     5
Name: count, dtype: int64

In [412]:
df_sel_final.to_csv(f'''{path}/copd_updated.csv''', index=False)



# Diabetes

In [413]:
df = pd.read_csv(f'''{P.codelist_path}/Diabetes_phenotype_PH152_ver_304_concepts_20231110T202045.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,C10F411,Type II diabetes mellitus with ulcer,Read codes v2,C1120,3379,Diabetes - Primary care,PH152,304,Diabetes,"[{""Disease"": ""Diabetes"", ""Medcode"": ""91646"", ""..."
1,C108912,Type 1 diabetes mellitus maturity onset,Read codes v2,C1120,3379,Diabetes - Primary care,PH152,304,Diabetes,"[{""Disease"": ""Diabetes"", ""Medcode"": ""97446"", ""..."
2,C10FQ00,Type 2 diabetes mellitus with exudative maculo...,Read codes v2,C1120,3379,Diabetes - Primary care,PH152,304,Diabetes,"[{""Disease"": ""Diabetes"", ""Medcode"": ""25591"", ""..."
3,C108.12,Type 1 diabetes mellitus,Read codes v2,C1120,3379,Diabetes - Primary care,PH152,304,Diabetes,"[{""Disease"": ""Diabetes"", ""Medcode"": ""17858"", ""..."
4,C10F611,Type II diabetes mellitus with retinopathy,Read codes v2,C1120,3379,Diabetes - Primary care,PH152,304,Diabetes,"[{""Disease"": ""Diabetes"", ""Medcode"": ""49655"", ""..."


In [414]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes       15
Med codes        135
Read codes v2    135
Name: code, dtype: int64

In [415]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2", "Med codes":"Med"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "Diabetes"
df["info"] = "PH152_v304"
df['vocab'].value_counts(dropna=False)

vocab
Read2    135
Med      135
ICD10     15
Name: count, dtype: int64

In [416]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Diabetes,C10F411,Read2,Type II diabetes mellitus with ulcer,PH152_v304
1,Diabetes,C108912,Read2,Type 1 diabetes mellitus maturity onset,PH152_v304
2,Diabetes,C10FQ00,Read2,Type 2 diabetes mellitus with exudative maculo...,PH152_v304
3,Diabetes,C108.12,Read2,Type 1 diabetes mellitus,PH152_v304
4,Diabetes,C10F611,Read2,Type II diabetes mellitus with retinopathy,PH152_v304


In [417]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    135
ICD10     15
Name: count, dtype: int64

In [418]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [419]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]

In [420]:
df_read_new.shape

(0, 5)

In [421]:
df_sel.to_csv(f'''{path}/diabetes.csv''', index=False)


# HF

In [422]:
df = pd.read_csv(f'''{P.codelist_path}/HF_phenotype_PH968_ver_2146_concepts_20231109T105756.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,10335000,Chronic right-sided heart failure (disorder),SNOMED CT codes,C2792,7313,CCU002_01 Heart failure - Phenotypes - England...,PH968,2146,CCU002_01 Heart failure,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
1,426263006,Congestive heart failure due to left ventricul...,SNOMED CT codes,C2792,7313,CCU002_01 Heart failure - Phenotypes - England...,PH968,2146,CCU002_01 Heart failure,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
2,420300004,New York Heart Association Classification - Cl...,SNOMED CT codes,C2792,7313,CCU002_01 Heart failure - Phenotypes - England...,PH968,2146,CCU002_01 Heart failure,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
3,134440006,Referral to heart failure clinic,SNOMED CT codes,C2792,7313,CCU002_01 Heart failure - Phenotypes - England...,PH968,2146,CCU002_01 Heart failure,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
4,83105008,Malignant hypertensive heart disease with cong...,SNOMED CT codes,C2792,7313,CCU002_01 Heart failure - Phenotypes - England...,PH968,2146,CCU002_01 Heart failure,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"


In [423]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes          4
Read codes v2       68
SNOMED  CT codes    41
Name: code, dtype: int64

In [424]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "HF"
df["info"] = "PH968_v2146"
df['vocab'].value_counts(dropna=False)

vocab
Read2     68
SNOMED    41
ICD10      4
Name: count, dtype: int64

In [425]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,HF,10335000,SNOMED,Chronic right-sided heart failure (disorder),PH968_v2146
1,HF,426263006,SNOMED,Congestive heart failure due to left ventricul...,PH968_v2146
2,HF,420300004,SNOMED,New York Heart Association Classification - Cl...,PH968_v2146
3,HF,134440006,SNOMED,Referral to heart failure clinic,PH968_v2146
4,HF,83105008,SNOMED,Malignant hypertensive heart disease with cong...,PH968_v2146


In [426]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    68
ICD10     4
Name: count, dtype: int64

In [427]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,HF,Read3,Left ventricular dysfunction monitoring first ...,PH968_v2146,Left ventricular dysfunction monitoring first ...,XaK6Y,Left ventricular dysfunction monitoring first ...
1,HF,Read3,Heart failure monitoring first letter ...,PH968_v2146,Heart failure monitoring first letter,XaMGs,Heart failure monitoring first letter
2,HF,Read3,Referred by heart failure nurse specialist ...,PH968_v2146,Referred by heart failure nurse specialist,XaLCj,Referred by heart failure nurse specialist
3,HF,Read3,New York Heart Association classification - cl...,PH968_v2146,New York Heart Association classification - cl...,XaJ9H,New York Heart Association classification - cl...
4,HF,Read3,H/O: Heart failure in last year ...,PH968_v2146,H/O: Heart failure in last year,XaBwi,H/O: Heart failure in last year


In [428]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]
df_read_new.shape

(66, 5)

In [429]:
df_sel_final = pd.concat([df_sel, df_read_new])
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
41,HF,I50,ICD10,Heart failure,PH968_v2146
42,HF,I110,ICD10,Hypertensive heart disease with (congestive) h...,PH968_v2146
43,HF,I130,ICD10,Hypertensive heart and renal disease with (con...,PH968_v2146
44,HF,I132,ICD10,Hypertensive heart and renal disease with both...,PH968_v2146
45,HF,9On0.,Read2,Left ventricular dysfunction monitoring first ...,PH968_v2146


In [430]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    68
Read3    66
ICD10     4
Name: count, dtype: int64

In [431]:
df_sel_final.to_csv(f'''{path}/hf_updated.csv''', index=False)


# HT

In [432]:
df = pd.read_csv(f'''{P.codelist_path}/HT_phenotype_PH970_ver_2148_concepts_20231115T192946.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,I11,Hypertensive heart disease,ICD10 codes,C2796,7317,CCU002_01 Hypertension - Phenotypes - England ...,PH970,2148,CCU002_01 Hypertension,"[{""terminology"": ""ICD10""}]"
1,I12,Hypertensive renal disease,ICD10 codes,C2796,7317,CCU002_01 Hypertension - Phenotypes - England ...,PH970,2148,CCU002_01 Hypertension,"[{""terminology"": ""ICD10""}]"
2,I15,Secondary hypertension,ICD10 codes,C2796,7317,CCU002_01 Hypertension - Phenotypes - England ...,PH970,2148,CCU002_01 Hypertension,"[{""terminology"": ""ICD10""}]"
3,I10,Essential (primary) hypertension,ICD10 codes,C2796,7317,CCU002_01 Hypertension - Phenotypes - England ...,PH970,2148,CCU002_01 Hypertension,"[{""terminology"": ""ICD10""}]"
4,I13,Hypertensive heart and renal disease,ICD10 codes,C2796,7317,CCU002_01 Hypertension - Phenotypes - England ...,PH970,2148,CCU002_01 Hypertension,"[{""terminology"": ""ICD10""}]"


In [433]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
BNF codes           849
ICD10 codes           5
Read codes v2        81
SNOMED  CT codes     74
Name: code, dtype: int64

In [434]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "HT"
df["info"] = "PH968_v2148"
df['vocab'].value_counts(dropna=False)

vocab
NaN       849
Read2      81
SNOMED     74
ICD10       5
Name: count, dtype: int64

In [435]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,HT,I11,ICD10,Hypertensive heart disease,PH968_v2148
1,HT,I12,ICD10,Hypertensive renal disease,PH968_v2148
2,HT,I15,ICD10,Secondary hypertension,PH968_v2148
3,HT,I10,ICD10,Essential (primary) hypertension,PH968_v2148
4,HT,I13,ICD10,Hypertensive heart and renal disease,PH968_v2148


In [436]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    81
ICD10     5
Name: count, dtype: int64

In [437]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,HT,Read3,Hypertensive retinopathy ...,PH968_v2148,Hypertensive retinopathy,F4211,Hypertensive retinopathy
1,HT,Read3,Secondary hypertension NOS ...,PH968_v2148,Secondary hypertension NOS,G24z.,Secondary hypertension NOS
2,HT,Read3,Hypertension six month review ...,PH968_v2148,Hypertension six month review,XaIyD,Hypertension six month review
3,HT,Read3,Patient on maximal tolerated antihypertensive ...,PH968_v2148,Patient on maximal tolerated antihypertensive ...,XaJ5h,Patient on maximal tolerated antihypertensive ...
4,HT,Read3,Hypertensive encephalopathy ...,PH968_v2148,Hypertensive encephalopathy,XE0VM,Hypertensive encephalopathy


In [438]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]
df_read_new.shape

(39, 5)

In [439]:
df_sel_final = pd.concat([df_sel, df_read_new])
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,HT,I11,ICD10,Hypertensive heart disease,PH968_v2148
1,HT,I12,ICD10,Hypertensive renal disease,PH968_v2148
2,HT,I15,ICD10,Secondary hypertension,PH968_v2148
3,HT,I10,ICD10,Essential (primary) hypertension,PH968_v2148
4,HT,I13,ICD10,Hypertensive heart and renal disease,PH968_v2148


In [440]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    81
Read3    39
ICD10     5
Name: count, dtype: int64

In [441]:
df_sel_final.to_csv(f'''{path}/ht_updated.csv''', index=False)


# Stroke + NOS

In [442]:
df_stroke = pd.read_csv(f'''{P.codelist_path}/phenotype_PH983_ver_2161_concepts_20231109T105206.csv''')
df_stroke.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,I60,Nontraumatic subarachnoid hemorrhage,ICD10 codes,C2822,7343,CCU002_01 Stroke - Phenotypes - England hospit...,PH983,2161,CCU002_01 Stroke,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,I61,Nontraumatic intracerebral hemorrhage,ICD10 codes,C2822,7343,CCU002_01 Stroke - Phenotypes - England hospit...,PH983,2161,CCU002_01 Stroke,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,230709005,Massive supratentorial cerebral hemorrhage (di...,SNOMED CT codes,C2823,7344,CCU002_01 Stroke - Phenotypes - England primar...,PH983,2161,CCU002_01 Stroke,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
3,195165005,Basal ganglia hemorrhage,SNOMED CT codes,C2823,7344,CCU002_01 Stroke - Phenotypes - England primar...,PH983,2161,CCU002_01 Stroke,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
4,276722003,Intracerebellar and posterior fossa hemorrhage...,SNOMED CT codes,C2823,7344,CCU002_01 Stroke - Phenotypes - England primar...,PH983,2161,CCU002_01 Stroke,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"


In [443]:
df_nos = pd.read_csv(f'''{P.codelist_path}/hdruk_pheno_library/phenotype_PH85_ver_170_concepts_20231109T105420.csv''')
df_nos.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,14A7.11,H/O: CVA,Read codes v2,C918,2975,Stroke NOS - Primary care,PH85,170,Stroke NOS,"[{""Disease"": ""Stroke NOS"", ""Medcode"": ""6305.0""..."
1,G66..00,Stroke and cerebrovascular accident unspecified,Read codes v2,C918,2975,Stroke NOS - Primary care,PH85,170,Stroke NOS,"[{""Disease"": ""Stroke NOS"", ""Medcode"": ""1469.0""..."
2,9h21.00,Excepted from stroke quality indicators: Patie...,Read codes v2,C918,2975,Stroke NOS - Primary care,PH85,170,Stroke NOS,"[{""Disease"": ""Stroke NOS"", ""Medcode"": ""11039.0..."
3,14A7.12,H/O: stroke,Read codes v2,C918,2975,Stroke NOS - Primary care,PH85,170,Stroke NOS,"[{""Disease"": ""Stroke NOS"", ""Medcode"": ""5871.0""..."
4,661M700,Stroke self-management plan agreed,Read codes v2,C918,2975,Stroke NOS - Primary care,PH85,170,Stroke NOS,"[{""Disease"": ""Stroke NOS"", ""Medcode"": ""107195...."


In [444]:
df_stroke.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes         14
Read codes v2       85
SNOMED  CT codes    96
Name: code, dtype: int64

In [445]:
df_nos.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes       8
Med codes        35
Read codes v2    35
Name: code, dtype: int64

In [446]:

df_stroke["vocab"] = df_stroke["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df_stroke["code_clean"] = df_stroke["code"]
df_stroke.loc[df_stroke["vocab"] == "ICD10", "code_clean"] = df_stroke["code_clean"].str.replace(".", "")
df_stroke.groupby(by="vocab").agg("count")["code"]


vocab
ICD10     14
Read2     85
SNOMED    96
Name: code, dtype: int64

In [447]:

df_nos["vocab"] = df_nos["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df_nos["code_clean"] = df_nos["code"]
df_nos.loc[df_nos["vocab"] == "ICD10", "code_clean"] = df_nos["code_clean"].str.replace(".", "")
df_nos.groupby(by="vocab").agg("count")["code"]

vocab
ICD10     8
Med      35
Read2    35
Name: code, dtype: int64

In [448]:
df_stroke_all = pd.concat([df_stroke, df_nos])

In [449]:
print(df_stroke.shape)
print(df_nos.shape)
print(df_stroke_all.shape)

(195, 12)
(78, 12)
(273, 12)


In [450]:

df_stroke_all["pheno_name"] = "Stroke"
df_stroke_all["info"] = "PH983_v2162_PH85_v170"
df_stroke_all['vocab'].value_counts(dropna=False)

vocab
Read2     120
SNOMED     96
Med        35
ICD10      22
Name: count, dtype: int64

In [451]:

df_stroke_sel = df_stroke_all[(df_stroke_all["vocab"]=="Read2") | (df_stroke_all["vocab"]=="ICD10")]
df_stroke_sel =df_stroke_sel[["pheno_name","code_clean", "vocab", "description", "info"]]

df_stroke_sel['vocab'].value_counts(dropna=False)


vocab
Read2    120
ICD10     22
Name: count, dtype: int64

In [452]:
df_stroke_sel.columns

Index(['pheno_name', 'code_clean', 'vocab', 'description', 'info'], dtype='object')

In [453]:
df_stroke_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Stroke,I60,ICD10,Nontraumatic subarachnoid hemorrhage,PH983_v2162_PH85_v170
1,Stroke,I61,ICD10,Nontraumatic intracerebral hemorrhage,PH983_v2162_PH85_v170
25,Stroke,G45,ICD10,Transient cerebral ischaemic attacks and relat...,PH983_v2162_PH85_v170
26,Stroke,H34,ICD10,Retinal vascular occlusions,PH983_v2162_PH85_v170
27,Stroke,I630,ICD10,Cerebral infarction,PH983_v2162_PH85_v170


In [454]:
# any CTV3?

df_read_new =  pd.merge(df_stroke_sel[(df_stroke_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,Stroke,Read3,Cereb infarct due unsp occlus/stenos precerebr...,PH983_v2162_PH85_v170,Cerebral infarction due to unspecified occlusi...,Gyu6G,[X]Cerebral infarction due to unspecified occl...
1,Stroke,Read3,Central post-stroke pain ...,PH983_v2162_PH85_v170,Central post-stroke pain,Xa0Ml,Central post-stroke pain
2,Stroke,Read3,CVA - cerebrovascular accident in the puerperi...,PH983_v2162_PH85_v170,Cerebrovascular disorders in the puerperium,XE2sl,Cerebrovascular disorders in the puerperium
3,Stroke,Read3,CVA - cerebrovascular accident in the puerperi...,PH983_v2162_PH85_v170,Cerebrovascular disorders in the puerperium,XM1PL,Stroke in the puerperium
4,Stroke,Read3,CVA - cerebrovascular accident in the puerperi...,PH983_v2162_PH85_v170,Cerebrovascular disorders in the puerperium,XM1PL,Stroke in the puerperium


In [455]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]
df_read_new.shape

(72, 5)

In [456]:
df_sel_final = pd.concat([df_stroke_sel, df_read_new])
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,Stroke,I60,ICD10,Nontraumatic subarachnoid hemorrhage,PH983_v2162_PH85_v170
1,Stroke,I61,ICD10,Nontraumatic intracerebral hemorrhage,PH983_v2162_PH85_v170
25,Stroke,G45,ICD10,Transient cerebral ischaemic attacks and relat...,PH983_v2162_PH85_v170
26,Stroke,H34,ICD10,Retinal vascular occlusions,PH983_v2162_PH85_v170
27,Stroke,I630,ICD10,Cerebral infarction,PH983_v2162_PH85_v170


In [457]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    120
Read3     72
ICD10     22
Name: count, dtype: int64

In [458]:
df_sel_final.to_csv(f'''{path}/stroke_updated.csv''', index=False)


# PE

In [459]:
df = pd.read_csv(f'''{P.codelist_path}/PE_phenotype_PH978_ver_2156_concepts_20231115T192721.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,I26.9,Pulmonary embolism,ICD10 codes,C2813,7334,CCU002_01 Pulmonary embolismm (PE) - Phenotype...,PH978,2156,CCU002_01 Pulmonary embolismm (PE),"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,I26.0,Pulmonary embolism,ICD10 codes,C2813,7334,CCU002_01 Pulmonary embolismm (PE) - Phenotype...,PH978,2156,CCU002_01 Pulmonary embolismm (PE),"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,438773007,Pulmonary embolism with mention of acute cor p...,SNOMED CT codes,C2814,7335,CCU002_01 Pulmonary embolismm (PE) - Phenotype...,PH978,2156,CCU002_01 Pulmonary embolismm (PE),"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
3,133971000119108,Pulmonary embolism with mention of acute cor p...,SNOMED CT codes,C2814,7335,CCU002_01 Pulmonary embolismm (PE) - Phenotype...,PH978,2156,CCU002_01 Pulmonary embolismm (PE),"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
4,7A0A1,Percutaneous transluminal embolectomy of pulmo...,Read codes v2,C2815,7336,CCU002_01 Pulmonary embolismm (PE) - Phenotype...,PH978,2156,CCU002_01 Pulmonary embolismm (PE),"[{""Category"": ""Procedure for Pulmonary embolis..."


In [460]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes         2
Read codes v2       8
SNOMED  CT codes    2
Name: code, dtype: int64

In [461]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "PE"
df["info"] = "PH978_v2156"
df['vocab'].value_counts(dropna=False)

vocab
Read2     8
ICD10     2
SNOMED    2
Name: count, dtype: int64

In [462]:
df_sel =df[["pheno_name","code_clean", "vocab", "description", "info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,PE,I269,ICD10,Pulmonary embolism,PH978_v2156
1,PE,I260,ICD10,Pulmonary embolism,PH978_v2156
2,PE,438773007,SNOMED,Pulmonary embolism with mention of acute cor p...,PH978_v2156
3,PE,133971000119108,SNOMED,Pulmonary embolism with mention of acute cor p...,PH978_v2156
4,PE,7A0A1,Read2,Percutaneous transluminal embolectomy of pulmo...,PH978_v2156


In [463]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)

vocab
Read2    8
ICD10    2
Name: count, dtype: int64

In [464]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,PE,Read3,Pulmonary thromboendarterectomy ...,PH978_v2156,Pulmonary thromboendarterectomy,XaLgy,Pulmonary thromboendarterectomy
1,PE,Read3,Pulmonary embolism ...,PH978_v2156,Pulmonary embolism,XE0Um,Pulmonary embolism
2,PE,Read3,Pulmonary embolism ...,PH978_v2156,Pulmonary embolism,G402.,Pulmonary infarction
3,PE,Read3,Pulmonary embolism ...,PH978_v2156,Pulmonary embolism,XE0Um,Pulmonary embolus
4,PE,Read3,Recurrent pulmonary embolism ...,PH978_v2156,Recurrent pulmonary embolism,XaOYV,Recurrent pulmonary embolism


In [465]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]
df_read_new.shape

(6, 5)

In [466]:
df_sel_final = pd.concat([df_sel, df_read_new])
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,PE,I269,ICD10,Pulmonary embolism,PH978_v2156
1,PE,I260,ICD10,Pulmonary embolism,PH978_v2156
4,PE,7A0A1,Read2,Percutaneous transluminal embolectomy of pulmo...,PH978_v2156
5,PE,7A093,Read2,Open embolectomy of pulmonary artery ...,PH978_v2156
6,PE,7A0B0,Read2,Pulmonary thromboendarterectomy ...,PH978_v2156


In [467]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    8
Read3    6
ICD10    2
Name: count, dtype: int64

In [468]:
df_sel_final.to_csv(f'''{path}/pe_updated.csv''', index=False)



# CKD

In [480]:
df_ckd_3_5 = pd.read_csv(f'''{P.codelist_path}/ckd/CKD_AD_Satges3_5_Dec2023.csv''')
df_ckd_3_5.head()

,Concept ID,Coding System,CKD Stage,Clinical Term
0,T82.4,ICD-10,5,mechanical complication of vascular dialysis c...
1,T86.1,ICD-10,5,kidney transplant failure and rejection
2,Y60.2,ICD-10,5,during kidney dialysis or other perfusion
3,Y61.2,ICD-10,5,during kidney dialysis or other perfusion
4,Y62.2,ICD-10,5,during kidney dialysis or other perfusion


In [481]:
df_ckd_3_5.groupby(by="Coding System").agg("count")["Concept ID"]


Coding System
ICD-10        17
Read-Code    131
SNOMED       103
Name: Concept ID, dtype: int64

In [482]:
df_ckd_3_5["vocab"] = df_ckd_3_5["Coding System"].map({"ICD-10": "ICD10", "Read-Code": "Read2"})
df_ckd_3_5["code"] = df_ckd_3_5["Concept ID"]
df_ckd_3_5["description"] = df_ckd_3_5["Clinical Term"]


In [483]:
df_ckd_3_5.groupby(by="vocab").agg("count")["code"]


vocab
ICD10     17
Read2    131
Name: code, dtype: int64

In [484]:
df_ckd_3_5.groupby(by="Coding System").agg("count")["Concept ID"]


Coding System
ICD-10        17
Read-Code    131
SNOMED       103
Name: Concept ID, dtype: int64

In [485]:
df_ckd_3_5.head()

,Concept ID,Coding System,CKD Stage,Clinical Term,vocab,code,description
0,T82.4,ICD-10,5,mechanical complication of vascular dialysis c...,ICD10,T82.4,mechanical complication of vascular dialysis c...
1,T86.1,ICD-10,5,kidney transplant failure and rejection,ICD10,T86.1,kidney transplant failure and rejection
2,Y60.2,ICD-10,5,during kidney dialysis or other perfusion,ICD10,Y60.2,during kidney dialysis or other perfusion
3,Y61.2,ICD-10,5,during kidney dialysis or other perfusion,ICD10,Y61.2,during kidney dialysis or other perfusion
4,Y62.2,ICD-10,5,during kidney dialysis or other perfusion,ICD10,Y62.2,during kidney dialysis or other perfusion


In [486]:

df_ckd_3_5["code_clean"] = df_ckd_3_5["code"]
df_ckd_3_5.loc[df_ckd_3_5["vocab"] == "ICD10", "code_clean"] = df_ckd_3_5["code_clean"].str.replace(".", "")
df_ckd_3_5["pheno_name"] = "CKD345"
df_ckd_3_5["info"] = "CCU003_01"
df_ckd_3_5['vocab'].value_counts(dropna=False)

vocab
Read2    131
NaN      103
ICD10     17
Name: count, dtype: int64

In [487]:
df_ckd_3_5_sel = df_ckd_3_5[["pheno_name","code_clean", "vocab", "description", "info"]]
df_ckd_3_5_sel.head()

,pheno_name,code_clean,vocab,description,info
0,CKD345,T824,ICD10,mechanical complication of vascular dialysis c...,CCU003_01
1,CKD345,T861,ICD10,kidney transplant failure and rejection,CCU003_01
2,CKD345,Y602,ICD10,during kidney dialysis or other perfusion,CCU003_01
3,CKD345,Y612,ICD10,during kidney dialysis or other perfusion,CCU003_01
4,CKD345,Y622,ICD10,during kidney dialysis or other perfusion,CCU003_01


In [488]:
df_ckd_3_5_sel = df_ckd_3_5_sel[(df_ckd_3_5_sel["vocab"]=="Read2") | (df_ckd_3_5_sel["vocab"]=="ICD10")]
df_ckd_3_5_sel['vocab'].value_counts(dropna=False)

vocab
Read2    131
ICD10     17
Name: count, dtype: int64

In [489]:
# any CTV3?

df_read_new =  pd.merge(df_ckd_3_5_sel[(df_ckd_3_5_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [490]:
df_ckd_3_5_sel.to_csv(f'''{path}/ckd345_updated.csv''', index=False)


# Atrial fibrilation


In [491]:
df = pd.read_csv(f'''{P.codelist_path}/AF_phenotype_PH36_ver_72_concepts_20231204T185155.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,9Os3.00,Atrial fibrillation monitoring verbal invite,Read codes v2,C789,2717,Atrial fibrillation - Primary care,PH36,72,Atrial fibrillation,"[{""Disease"": ""Atrial fibrillation"", ""Medcode"":..."
1,G573000,Atrial fibrillation,Read codes v2,C789,2717,Atrial fibrillation - Primary care,PH36,72,Atrial fibrillation,"[{""Disease"": ""Atrial fibrillation"", ""Medcode"":..."
2,14AR.00,History of atrial flutter,Read codes v2,C789,2717,Atrial fibrillation - Primary care,PH36,72,Atrial fibrillation,"[{""Disease"": ""Atrial fibrillation"", ""Medcode"":..."
3,212R.00,Atrial fibrillation resolved,Read codes v2,C789,2717,Atrial fibrillation - Primary care,PH36,72,Atrial fibrillation,"[{""Disease"": ""Atrial fibrillation"", ""Medcode"":..."
4,9Os0.00,Atrial fibrillation monitoring first letter,Read codes v2,C789,2717,Atrial fibrillation - Primary care,PH36,72,Atrial fibrillation,"[{""Disease"": ""Atrial fibrillation"", ""Medcode"":..."


In [492]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes       1
Med codes        23
Read codes v2    23
Name: code, dtype: int64

In [493]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "AF"
df["info"] = "PH36_v72"
df['vocab'].value_counts(dropna=False)

vocab
Read2    23
Med      23
ICD10     1
Name: count, dtype: int64

In [494]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,AF,9Os3.00,Read2,Atrial fibrillation monitoring verbal invite,PH36_v72
1,AF,G573000,Read2,Atrial fibrillation,PH36_v72
2,AF,14AR.00,Read2,History of atrial flutter,PH36_v72
3,AF,212R.00,Read2,Atrial fibrillation resolved,PH36_v72
4,AF,9Os0.00,Read2,Atrial fibrillation monitoring first letter,PH36_v72


In [495]:
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]
df_sel['vocab'].value_counts(dropna=False)

vocab
Read2    23
ICD10     1
Name: count, dtype: int64

In [496]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [497]:
df_sel.to_csv(f'''{path}/af.csv''', index=False)


# DVT

In [498]:
df = pd.read_csv(f'''{P.codelist_path}/dvt_phenotype_PH966_ver_2144_concepts_20231204T190037.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,I80,Phlebitis and thrombophlebitis,ICD10 codes,C2783,7304,"CCU002_01 Deep vein thrombosis, intracranial v...",PH966,2144,"CCU002_01 Deep vein thrombosis, intracranial v...","[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,I67.6,Intracranial venous thrombosis,ICD10 codes,C2784,7305,"CCU002_01 Deep vein thrombosis, intracranial v...",PH966,2144,"CCU002_01 Deep vein thrombosis, intracranial v...","[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,G08,Intracranial venous thrombosis,ICD10 codes,C2784,7305,"CCU002_01 Deep vein thrombosis, intracranial v...",PH966,2144,"CCU002_01 Deep vein thrombosis, intracranial v...","[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
3,I63.6,Intracranial venous thrombosis,ICD10 codes,C2784,7305,"CCU002_01 Deep vein thrombosis, intracranial v...",PH966,2144,"CCU002_01 Deep vein thrombosis, intracranial v...","[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
4,195230003,Cerebral infarction due to cerebral venous thr...,SNOMED CT codes,C2785,7306,"CCU002_01 Deep vein thrombosis, intracranial v...",PH966,2144,"CCU002_01 Deep vein thrombosis, intracranial v...","[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"


In [499]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes         16
Read codes v2        1
SNOMED  CT codes     1
Name: code, dtype: int64

In [500]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "DVT"
df["info"] = "PH966_v2144"
df['vocab'].value_counts(dropna=False)

vocab
ICD10     16
SNOMED     1
Read2      1
Name: count, dtype: int64

In [501]:
df_sel =df[["pheno_name","code_clean", "vocab", "description", "info"]]
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]

df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,DVT,I80,ICD10,Phlebitis and thrombophlebitis,PH966_v2144
1,DVT,I676,ICD10,Intracranial venous thrombosis,PH966_v2144
2,DVT,G08,ICD10,Intracranial venous thrombosis,PH966_v2144
3,DVT,I636,ICD10,Intracranial venous thrombosis,PH966_v2144
5,DVT,O879,ICD10,Thrombosis during pregnancy and puerperium,PH966_v2144


In [502]:
df_sel['vocab'].value_counts(dropna=False)


vocab
ICD10    16
Read2     1
Name: count, dtype: int64

In [503]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [504]:
df_sel.to_csv(f'''{path}/dvt.csv''', index=False)



# PAD

In [505]:
df = pd.read_csv(f'''{P.codelist_path}/pad_phenotype_PH236_ver_472_concepts_20231204T190615.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,7A12312,Dacron aortoiliac Y graft,Read codes v2,C1349,3837,Peripheral arterial disease - Primary care,PH236,472,Peripheral arterial disease,"[{""Disease"": ""Peripheral arterial disease"", ""M..."
1,7A49.11,Reconstruction of common femoral artery,Read codes v2,C1349,3837,Peripheral arterial disease - Primary care,PH236,472,Peripheral arterial disease,"[{""Disease"": ""Peripheral arterial disease"", ""M..."
2,7A48200,Bypass femoral artery by fem/pop art anast c v...,Read codes v2,C1349,3837,Peripheral arterial disease - Primary care,PH236,472,Peripheral arterial disease,"[{""Disease"": ""Peripheral arterial disease"", ""M..."
3,G73yz00,Other specified peripheral vascular disease NOS,Read codes v2,C1349,3837,Peripheral arterial disease - Primary care,PH236,472,Peripheral arterial disease,"[{""Disease"": ""Peripheral arterial disease"", ""M..."
4,7A48800,Bypass femoral artery by fem/peron a anast c p...,Read codes v2,C1349,3837,Peripheral arterial disease - Primary care,PH236,472,Peripheral arterial disease,"[{""Disease"": ""Peripheral arterial disease"", ""M..."


In [506]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes        6
Med codes        201
OPCS4 codes       71
Read codes v2    201
Name: code, dtype: int64

In [507]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "PAD"
df["info"] = "PH236_v472"
df['vocab'].value_counts(dropna=False)

vocab
Read2    201
Med      201
NaN       71
ICD10      6
Name: count, dtype: int64

In [508]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]

df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,PAD,7A12312,Read2,Dacron aortoiliac Y graft,PH236_v472
1,PAD,7A49.11,Read2,Reconstruction of common femoral artery,PH236_v472
2,PAD,7A48200,Read2,Bypass femoral artery by fem/pop art anast c v...,PH236_v472
3,PAD,G73yz00,Read2,Other specified peripheral vascular disease NOS,PH236_v472
4,PAD,7A48800,Read2,Bypass femoral artery by fem/peron a anast c p...,PH236_v472


In [509]:
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    201
ICD10      6
Name: count, dtype: int64

In [510]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [511]:
df_sel.to_csv(f'''{path}/pad.csv''', index=False)


# Depression

In [512]:
df = pd.read_csv(f'''{P.codelist_path}/depression_phenotype_PH964_ver_2142_concepts_20231204T190936.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,F32,Depressive episode,ICD10 codes,C2778,7299,CCU002_01 Depression - Phenotypes - England ho...,PH964,2142,CCU002_01 Depression,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,F33,Recurrent depressive disorder,ICD10 codes,C2778,7299,CCU002_01 Depression - Phenotypes - England ho...,PH964,2142,CCU002_01 Depression,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,231499006,Endogenous depression first episode (disorder),SNOMED CT codes,C2779,7300,CCU002_01 Depression - Phenotypes - England pr...,PH964,2142,CCU002_01 Depression,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
3,370143000,Major depressive disorder (disorder),SNOMED CT codes,C2779,7300,CCU002_01 Depression - Phenotypes - England pr...,PH964,2142,CCU002_01 Depression,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"
4,755321000000106,"Single major depressive episode, severe, with ...",SNOMED CT codes,C2779,7300,CCU002_01 Depression - Phenotypes - England pr...,PH964,2142,CCU002_01 Depression,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"


In [513]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes          2
Read codes v2       75
SNOMED  CT codes    67
Name: code, dtype: int64

In [514]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "Depression"
df["info"] = "PH964_2142"
df['vocab'].value_counts(dropna=False)

vocab
Read2     75
SNOMED    67
ICD10      2
Name: count, dtype: int64

In [515]:
df_sel =df[["pheno_name","code_clean", "vocab","description", "info"]]
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]

df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Depression,F32,ICD10,Depressive episode,PH964_2142
1,Depression,F33,ICD10,Recurrent depressive disorder,PH964_2142
69,Depression,Eu33y,Read2,[X]Other recurrent depressive disorders ...,PH964_2142
70,Depression,9H91.,Read2,Depression medication review ...,PH964_2142
71,Depression,Eu251,Read2,"[X]Schizoaffective disorder, depressive type ...",PH964_2142


In [516]:
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    75
ICD10     2
Name: count, dtype: int64

In [517]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,Depression,Read3,Depression medication review ...,PH964_2142,Depression medication review,XaK6e,Depression medication review
1,Depression,Read3,"[X]Schizoaffective disorder, depressive type ...",PH964_2142,"[X]Schizoaffective disorder, depressive type",XE2un,"Schizoaffective disorder, depressive type"
2,Depression,Read3,"[X]Schizoaffective disorder, depressive type ...",PH964_2142,"[X]Schizoaffective disorder, depressive type",XE2un,"Schizoaffective disorder, depressive type"
3,Depression,Read3,"[X]Schizoaffective disorder, depressive type ...",PH964_2142,"[X]Schizoaffective disorder, depressive type",XE2un,"Schizophreniform psychosis, depressive type"
4,Depression,Read3,"[X]Schizoaffective disorder, depressive type ...",PH964_2142,"[X]Schizoaffective disorder, depressive type",XE2un,"Schizoaffective disorder, depressive type"


In [518]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]
df_read_new.shape

(106, 5)

In [519]:
df_sel_final = pd.concat([df_sel, df_read_new])
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,Depression,F32,ICD10,Depressive episode,PH964_2142
1,Depression,F33,ICD10,Recurrent depressive disorder,PH964_2142
69,Depression,Eu33y,Read2,[X]Other recurrent depressive disorders ...,PH964_2142
70,Depression,9H91.,Read2,Depression medication review ...,PH964_2142
71,Depression,Eu251,Read2,"[X]Schizoaffective disorder, depressive type ...",PH964_2142


In [520]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read3    106
Read2     75
ICD10      2
Name: count, dtype: int64

In [521]:
df_sel_final.to_csv(f'''{path}/depression_updated.csv''', index=False)


# Obesity

In [522]:
df = pd.read_csv(f'''{P.codelist_path}/obesity_phenotype_PH221_ver_442_concepts_20231205T120131.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,C38z000,Simple obesity NOS,Read codes v2,C1314,3767,Obesity - Primary care,PH221,442,Obesity,"[{""Disease"": ""Obesity"", ""Medcode"": ""11401.0"", ..."
1,66C..00,Obesity monitoring,Read codes v2,C1314,3767,Obesity - Primary care,PH221,442,Obesity,"[{""Disease"": ""Obesity"", ""Medcode"": ""11461.0"", ..."
2,C380000,Obesity due to excess calories,Read codes v2,C1314,3767,Obesity - Primary care,PH221,442,Obesity,"[{""Disease"": ""Obesity"", ""Medcode"": ""38799.0"", ..."
3,C380400,Central obesity,Read codes v2,C1314,3767,Obesity - Primary care,PH221,442,Obesity,"[{""Disease"": ""Obesity"", ""Medcode"": ""22695.0"", ..."
4,7613600.0,Maintenance of gastric band,Read codes v2,C1314,3767,Obesity - Primary care,PH221,442,Obesity,"[{""Disease"": ""Obesity"", ""Medcode"": ""97691.0"", ..."


In [523]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes       1
Med codes        70
Read codes v2    70
Name: code, dtype: int64

In [524]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "Obesity"
df["info"] = "PH221_442"
df['vocab'].value_counts(dropna=False)

vocab
Read2    70
Med      70
ICD10     1
Name: count, dtype: int64

In [525]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]

df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Obesity,C38z000,Read2,Simple obesity NOS,PH221_442
1,Obesity,66C..00,Read2,Obesity monitoring,PH221_442
2,Obesity,C380000,Read2,Obesity due to excess calories,PH221_442
3,Obesity,C380400,Read2,Central obesity,PH221_442
4,Obesity,7613600.0,Read2,Maintenance of gastric band,PH221_442


In [526]:
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    70
ICD10     1
Name: count, dtype: int64

In [527]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC


In [528]:
df_sel.to_csv(f'''{path}/obesity.csv''', index=False)


# Dementia


In [529]:
df = pd.read_csv(f'''{P.codelist_path}/dementia_phenotype_PH963_ver_2141_concepts_20231205T122311.csv''')
df.head()

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,F00,Dementia in Alzheimer's disease,ICD10 codes,C2773,7294,CCU002_01 Dementia - Phenotypes - England hosp...,PH963,2141,CCU002_01 Dementia,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
1,F03,Unspecified dementia,ICD10 codes,C2773,7294,CCU002_01 Dementia - Phenotypes - England hosp...,PH963,2141,CCU002_01 Dementia,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
2,G30,Alzheimer's disease,ICD10 codes,C2773,7294,CCU002_01 Dementia - Phenotypes - England hosp...,PH963,2141,CCU002_01 Dementia,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
3,F05.1,Delirium superimposed on dementia,ICD10 codes,C2773,7294,CCU002_01 Dementia - Phenotypes - England hosp...,PH963,2141,CCU002_01 Dementia,"[{""terminology"": ""ICD10"", ""code_type"": ""1""}]"
4,419261000000107,"[X]Dementia in Alzheimer's dis, atypical or mi...",SNOMED CT codes,C2774,7295,CCU002_01 Dementia - Phenotypes - England prim...,PH963,2141,CCU002_01 Dementia,"[{""terminology"": ""SNOMEDCT"", ""code_type"": ""1""}]"


In [530]:
df.groupby(by="coding_system").agg("count")["code"]


coding_system
ICD10 codes          5
Read codes v2       54
SNOMED  CT codes    91
Name: code, dtype: int64

In [531]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "Dementia"
df["info"] = "PH963_2141"
df['vocab'].value_counts(dropna=False)

vocab
SNOMED    91
Read2     54
ICD10      5
Name: count, dtype: int64

In [532]:
df_sel =df[["pheno_name","code_clean", "vocab", "description","info"]]
df_sel = df_sel[(df_sel["vocab"]=="Read2") | (df_sel["vocab"]=="ICD10")]

df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Dementia,F00,ICD10,Dementia in Alzheimer's disease,PH963_2141
1,Dementia,F03,ICD10,Unspecified dementia,PH963_2141
2,Dementia,G30,ICD10,Alzheimer's disease,PH963_2141
3,Dementia,F051,ICD10,Delirium superimposed on dementia,PH963_2141
83,Dementia,F01,ICD10,Vascular dementia,PH963_2141


In [533]:
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    54
ICD10     5
Name: count, dtype: int64

In [535]:
df_sel.to_csv(f'''{path}/dementia.csv''', index=False)


In [536]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,info,READV2_DESC,code_clean,TERMV3_DESC
0,Dementia,Read3,[X]Mixed cortical and subcortical vascular dem...,PH963_2141,[X]Mixed cortical and subcortical vascular dem...,X003V,Mixed cortical and subcortical vascular dementia
1,Dementia,Read3,Dementia monitoring telephone invite ...,PH963_2141,Dementia monitoring telephone invite,XaMGK,Dementia monitoring telephone invite
2,Dementia,Read3,[X]Dementia in Alzheimer's disease with late o...,PH963_2141,[X]Dementia in Alzheimer's disease with late o...,X0030,Dementia in Alzheimer's disease with late onset
3,Dementia,Read3,[X]Dementia in Alzheimer's disease with late o...,PH963_2141,[X]Dementia in Alzheimer's disease with late o...,X0030,Dementia in Alzheimer's disease with late onset
4,Dementia,Read3,[X]Dementia in Alzheimer's disease with late o...,PH963_2141,[X]Dementia in Alzheimer's disease with late o...,X0030,"SDAT - Senile dementia, Alzheimer's type"


In [537]:
df_read_new = df_read_new[['pheno_name', 'code_clean', 'vocab', 'description', 'info']]
df_read_new.shape

(54, 5)

In [538]:
df_sel_final = pd.concat([df_sel, df_read_new])
df_sel_final.head()

,pheno_name,code_clean,vocab,description,info
0,Dementia,F00,ICD10,Dementia in Alzheimer's disease,PH963_2141
1,Dementia,F03,ICD10,Unspecified dementia,PH963_2141
2,Dementia,G30,ICD10,Alzheimer's disease,PH963_2141
3,Dementia,F051,ICD10,Delirium superimposed on dementia,PH963_2141
83,Dementia,F01,ICD10,Vascular dementia,PH963_2141


In [539]:
df_sel_final['vocab'].value_counts(dropna=False)


vocab
Read2    54
Read3    54
ICD10     5
Name: count, dtype: int64

In [540]:
df_sel_final.to_csv(f'''{path}/dementia_updated.csv''', index=False)


In [541]:
df_sel.head()

,pheno_name,code_clean,vocab,description,info
0,Dementia,F00,ICD10,Dementia in Alzheimer's disease,PH963_2141
1,Dementia,F03,ICD10,Unspecified dementia,PH963_2141
2,Dementia,G30,ICD10,Alzheimer's disease,PH963_2141
3,Dementia,F051,ICD10,Delirium superimposed on dementia,PH963_2141
83,Dementia,F01,ICD10,Vascular dementia,PH963_2141


# Smoking

In [542]:
df = pd.read_csv(f'''{P.codelist_path}/smoking/phenotype_PH350_ver_1513_concepts_20231224T173857.csv''')
df.head(100)

,code,description,coding_system,concept_id,concept_version_id,concept_name,phenotype_id,phenotype_version_id,phenotype_name,code_attributes
0,137K.00,Stopped smoking,Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex smoker""}]"
1,137O.00,Ex cigar smoker,Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex smoker""}]"
2,1379,Ex-moderate smoker (10-19/day),Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex smoker""}]"
3,137S.00,Ex smoker,Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex smoker""}]"
4,137N.00,Ex pipe smoker,Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex smoker""}]"
5,1377,Ex-trivial smoker (<1/day),Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex smoker""}]"
6,137T.00,Date ceased smoking,Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex smoker""}]"
7,ZV11600,[V]Personal history of tobacco abuse,Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex or current smoker""}]"
8,H310100,Smokers' cough,Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Ex or current smoker""}]"
9,137L.00,Current non-smoker,Read codes v2,C1678,5500,Smoking Status - Primary Care,PH350,1513,Smoking Status,"[{""CALIBER category"": ""Non smoker""}]"


In [543]:
import json
df['status'] = df['code_attributes'].apply(lambda x: json.loads(x)[0].get("CALIBER category"))

In [545]:
df.groupby("status").size().reset_index(name="Counts")

,status,Counts
0,Current smoker,3
1,Ex or current smoker,2
2,Ex smoker,14
3,Non smoker,5


In [546]:

df["vocab"] = df["coding_system"].map({"ICD10 codes": "ICD10", "SNOMED  CT codes": "SNOMED", "Read codes v2": "Read2","BFF code":"BNF", "Med codes":"Med"})
df["code_clean"] = df["code"]
df.loc[df["vocab"] == "ICD10", "code_clean"] = df["code_clean"].str.replace(".", "")
df["pheno_name"] = "smoking"
df["info"] = "PH350_1513"
df['vocab'].value_counts(dropna=False)

vocab
Read2    21
ICD10     3
Name: count, dtype: int64

In [548]:
df_sel = df[["pheno_name", "code_clean", "vocab", "description", "status", "info"]]
df_sel

,pheno_name,code_clean,vocab,description,status,info
0,smoking,137K.00,Read2,Stopped smoking,Ex smoker,PH350_1513
1,smoking,137O.00,Read2,Ex cigar smoker,Ex smoker,PH350_1513
2,smoking,1379,Read2,Ex-moderate smoker (10-19/day),Ex smoker,PH350_1513
3,smoking,137S.00,Read2,Ex smoker,Ex smoker,PH350_1513
4,smoking,137N.00,Read2,Ex pipe smoker,Ex smoker,PH350_1513
5,smoking,1377,Read2,Ex-trivial smoker (<1/day),Ex smoker,PH350_1513
6,smoking,137T.00,Read2,Date ceased smoking,Ex smoker,PH350_1513
7,smoking,ZV11600,Read2,[V]Personal history of tobacco abuse,Ex or current smoker,PH350_1513
8,smoking,H310100,Read2,Smokers' cough,Ex or current smoker,PH350_1513
9,smoking,137L.00,Read2,Current non-smoker,Non smoker,PH350_1513


In [549]:
df_sel['vocab'].value_counts(dropna=False)


vocab
Read2    21
ICD10     3
Name: count, dtype: int64

In [550]:
df_sel['status'].value_counts(dropna=False)


status
Ex smoker               14
Non smoker               5
Current smoker           3
Ex or current smoker     2
Name: count, dtype: int64

In [551]:
# any CTV3?

df_read_new =  pd.merge(df_sel[(df_sel["vocab"]=="Read2") ], df_read_diff, on= ["code_clean"], how="inner")
df_read_new = df_read_new.drop(columns=["code_clean"])
df_read_new = df_read_new.rename(columns={"READ3":"code_clean"})
df_read_new["vocab"] = "Read3"

df_read_new.head()

,pheno_name,vocab,description,status,info,READV2_DESC,code_clean,TERMV3_DESC


In [552]:
df_sel.to_csv(f'''{path}/smoking.csv''', index=False)
